# 中文情緒分類：預測與訓練圖表

載入 `finetune.ipynb` 產生的模型，預測中文句子的情緒，並繪製訓練／驗證 Loss 與驗證 Macro F1 曲線。
本 notebook 可在獨立的 Colab 執行階段執行，不需保留訓練時的 Python 變數。

## 快速開始

1. 先執行 `finetune.ipynb` 完成訓練。
2. 將此 notebook 上傳至 Colab，從上往下執行；GPU 可加速預測，CPU 也能執行。
3. 訓練時使用 Drive，請將 `USE_GOOGLE_DRIVE` 設為 `True`，並使用相同專案位置。
4. 使用暫存空間時，若找不到模型，程式會請你上傳訓練完成後下載的 `bert_artifacts.zip`。
5. 修改 `TEST_TEXTS` 為要分類的句子，執行預測後即可查看結果表格與圖表。

## 輸出

| 檔案 | 內容 |
| --- | --- |
| `output/predictions.csv` | 輸入句子、預測情緒、信心分數 |
| `output/plots/loss_curve.png` | 訓練與驗證 Loss 曲線 |
| `output/plots/f1_curve.png` | 驗證 Macro F1 曲線 |

所有圖表也會直接顯示在 notebook 中。預測標籤由模型的 `config.json` 讀取，不需手動指定類別編號。


## 1. 安裝與匯入套件

安裝模型推論、表格與繪圖所需套件。PyTorch 使用 Colab 提供的環境。
使用與訓練 notebook 相同的 Transformers 5.20.0 與 `huggingface-hub>=1.31,<2.0`。
首次安裝或變更套件版本後，請選擇「執行階段 → 重新啟動工作階段」，再從下方匯入套件儲存格繼續。


In [ ]:
%pip install -q --upgrade "transformers==5.20.0" "huggingface-hub>=1.31,<2.0" matplotlib pandas


In [ ]:
import json
from io import BytesIO
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline


## 2. 設定專案位置

選擇一種儲存方式，兩份 notebook 必須使用相同設定：

| 設定 | 專案位置 | 使用方式 |
| --- | --- | --- |
| `USE_GOOGLE_DRIVE = False` | `/content/HW_bert_finetune` | 使用 Colab 暫存空間，以壓縮檔帶走模型與紀錄 |
| `USE_GOOGLE_DRIVE = True` | `/content/drive/MyDrive/HW_bert_finetune` | 掛載 Google Drive，將資料與輸出直接存入雲端硬碟 |

使用 Drive 時，執行下方儲存格並完成授權。若要更換資料夾，可修改 `PROJECT_DIR`。
Colab 暫存空間會隨執行階段刪除，請在結束前下載產出，或使用 Drive 儲存。
詳見 [Colab 說明](https://research.google.com/colaboratory/faq.html)。


In [ ]:
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/HW_bert_finetune")
else:
    PROJECT_DIR = Path("/content/HW_bert_finetune") if Path("/content").is_dir() else Path.cwd()
    if (PROJECT_DIR / "HW_bert_finetune").is_dir():
        PROJECT_DIR /= "HW_bert_finetune"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"專案位置：{PROJECT_DIR}")


In [ ]:
MODEL_DIR = PROJECT_DIR / "output"
STATE_FILE = MODEL_DIR / "trainer_state.json"
RESULT_DIR = PROJECT_DIR / "output"
PLOT_DIR = RESULT_DIR / "plots"
PREDICTION_FILE = RESULT_DIR / "predictions.csv"


## 3. 準備模型與訓練紀錄

專案目錄已有模型時，直接使用現有檔案。
若找不到模型，請在上傳視窗選擇 `finetune.ipynb` 下載的 `bert_artifacts.zip`，程式會還原專案內的資料夾結構。

`MODEL_DIR` 應包含 `config.json`、模型權重（例如 `model.safetensors`）與 tokenizer 檔案。
繪圖另需要同目錄的 `trainer_state.json`；若缺少訓練紀錄，仍可執行句子預測。
自訂模型路徑時，請同時確認 `STATE_FILE` 指向對應的訓練紀錄。


In [ ]:
if not (MODEL_DIR / "config.json").is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"找不到 {MODEL_DIR}，請先執行 finetune.ipynb 或還原訓練成果。") from None
    print("請上傳 bert_artifacts.zip")
    uploaded = files.upload()
    if "bert_artifacts.zip" not in uploaded:
        raise FileNotFoundError("未收到 bert_artifacts.zip，請重新執行此儲存格並選擇該檔案。")
    with ZipFile(BytesIO(uploaded["bert_artifacts.zip"])) as archive:
        root = PROJECT_DIR.resolve()
        if any(not (root / name).resolve().is_relative_to(root) for name in archive.namelist()):
            raise ValueError("壓縮檔內含不合法的相對路徑。")
        archive.extractall(root)
    del uploaded

if not (MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(f"壓縮檔內找不到模型設定，請確認 MODEL_DIR：{MODEL_DIR}")
print(f"模型位置：{MODEL_DIR}")


## 4. 載入情緒分類模型

從本機專案目錄載入訓練完成的模型與 tokenizer，並自動選擇 GPU 或 CPU。
情緒名稱與整數 ID 由模型設定讀取，確保預測結果與訓練標籤相符。


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True)
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)
print(f"預測裝置：{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"情緒類別：{model.config.id2label}")


## 5. 輸入句子與預測

修改下方 `TEST_TEXTS`，每個字串代表一句要分類的文字。
模型會選出分數最高的情緒；`score` 為 softmax 信心分數，範圍介於 0 與 1。
表格會顯示在此處，完整數值另存為 `output/predictions.csv`。

詳見 [Hugging Face 文字分類文件](https://huggingface.co/docs/transformers/en/main_classes/pipelines#transformers.TextClassificationPipeline)。


In [ ]:
TEST_TEXTS = [
    "我每天都能跟她一起上學，我好開心！",
    "最好的朋友要離開臺灣了，以後可能不容易再見面...",
    "我覺得我快不行了...",
    "剛剛收到研究所錄取的通知書！",
    "今年的冬天好像比較晚來。",
]
PREDICT_BATCH_SIZE = 16
MAX_SEQ_LENGTH = 512


In [ ]:
if not TEST_TEXTS or any(not isinstance(text, str) or not text.strip() for text in TEST_TEXTS):
    raise ValueError("TEST_TEXTS 至少需要一句非空白文字，且每一項都必須是字串。")
results = classifier(
    TEST_TEXTS,
    batch_size=PREDICT_BATCH_SIZE,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
    function_to_apply="softmax",
)
predictions = pd.DataFrame(
    [{"text": text, "emotion": result["label"], "score": result["score"]}
     for text, result in zip(TEST_TEXTS, results)]
)
display(predictions.style.format({"score": "{:.4f}"}))
RESULT_DIR.mkdir(parents=True, exist_ok=True)
predictions.to_csv(PREDICTION_FILE, index=False, encoding="utf-8-sig")
print(f"預測結果：{PREDICTION_FILE}")


## 6. 繪製訓練曲線

讀取 `trainer_state.json` 的 `log_history`，橫軸為 optimizer step。

- Loss 圖顯示訓練 `loss` 與驗證 `eval_loss`，數值越低代表分類誤差越小。
- F1 圖顯示驗證 `eval_f1`（Macro F1），數值越接近 1 表示分類表現越好。

圖表會直接顯示並存入 `output/plots/`。若缺少某項指標，只略過對應曲線；
完全沒有紀錄時會顯示提示，不影響已產生的句子預測結果。


In [ ]:
def plot_history(logs, metrics, title, ylabel, filename):
    """從 Trainer 紀錄擷取指標，顯示曲線並儲存 PNG。"""
    fig, ax = plt.subplots(figsize=(10, 5))
    for key, label in metrics.items():
        points = [(row["step"], row[key]) for row in logs if "step" in row and key in row]
        if points:
            steps, values = zip(*points)
            ax.plot(steps, values, marker="o" if key.startswith("eval_") else None, label=label)
    if not ax.lines:
        plt.close(fig)
        print(f"略過 {title}：沒有對應的指標紀錄。")
        return
    ax.set(xlabel="Optimizer Step", ylabel=ylabel, title=title)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    path = PLOT_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"圖表：{path}")
    return path


generated_plots = []
if not STATE_FILE.is_file():
    print(f"無法繪圖：找不到 {STATE_FILE}。請確認已執行 finetune.ipynb 的模型儲存步驟。")
else:
    state = json.loads(STATE_FILE.read_text(encoding="utf-8"))
    logs = state.get("log_history", [])
    for metrics, title, ylabel, filename in (
        ({"loss": "Training Loss", "eval_loss": "Validation Loss"},
         "BERT Training and Validation Loss", "Loss", "loss_curve.png"),
        ({"eval_f1": "Validation Macro F1"},
         "BERT Validation Macro F1", "Macro F1", "f1_curve.png"),
    ):
        path = plot_history(logs, metrics, title, ylabel, filename)
        if path is not None:
            generated_plots.append(path)


## 7. 下載預測結果與圖表

將預測 CSV 與本次可用的圖表打包成 `prediction_results.zip`。
預設不會自動下載；需要本機副本時，將 `DOWNLOAD_RESULTS` 改為 `True` 再執行此儲存格。
Drive 模式也可直接在雲端硬碟的 `output/` 查看檔案。


In [ ]:
DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS:
    result_files = [PREDICTION_FILE, *generated_plots]
    archive_path = RESULT_DIR / "prediction_results.zip"
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for path in result_files:
            if path.is_file():
                archive.write(path, arcname=path.relative_to(RESULT_DIR).as_posix())
    print(f"預測成果壓縮檔：{archive_path}")
    try:
        from google.colab import files
    except ImportError:
        pass
    else:
        files.download(str(archive_path))
